# VAD Storage

> Standardized SQLite storage for voice-activity-detection results with content hashing.

**Born-final (execution stage 8 — Option C):** the cache/persistence bookend the `GenericVADAdapter` wraps around the pure-compute VAD tool. It is the task-keyed successor to the fused-era `MediaAnalysisStorage` analysis path (silero-vad was the sole `MediaAnalysisPlugin`): keyed `(file_path, config_hash)`, content-correct on `file_hash`. No `job_id` (VAD never carried one) and no pre-cache migration machinery — the adapter DB (`<CAPABILITY_DATA_DIR>/vad.db`) is brand-new, so it starts with the clean UNIQUE-keyed schema.

In [ ]:
#| default_exp storage

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
import logging
import sqlite3
import time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

from cjm_substrate.utils.hashing import hash_file
# CR-14 follow-up: storage events are substrate-family accounts. The helpers
# below RECORD them (no-op outside a worker call span); the host journals
# them off the response header with worker_reported=True.
from cjm_substrate.core.journal_store import SubstrateEventType
from cjm_substrate.core.wire import record_account

In [ ]:
#| export
@dataclass
class VADRow:
    """A single row from the vad_results table."""
    file_path: str       # Path to the analyzed (model-ready) audio file
    file_hash: str       # Hash of the analyzed file in "algo:hexdigest" format
    config_hash: str     # Hash of the VAD detection config used
    ranges: Optional[List[Dict[str, Any]]] = None  # Detected speech segments (serialized TimeRanges)
    metadata: Optional[Dict[str, Any]] = None       # VAD metadata
    created_at: Optional[float] = None               # Unix timestamp

In [ ]:
#| export
class VADStorage:
    """Standardized SQLite storage for voice-activity-detection results."""

    SCHEMA = """
        CREATE TABLE IF NOT EXISTS vad_results (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            file_path TEXT NOT NULL,
            file_hash TEXT NOT NULL,
            config_hash TEXT NOT NULL,
            ranges JSON,
            metadata JSON,
            created_at REAL NOT NULL,
            UNIQUE(file_path, config_hash)
        )
    """

    INDEX = "CREATE INDEX IF NOT EXISTS idx_vad_file_path ON vad_results(file_path);"

    def __init__(
        self,
        db_path: str  # Absolute path to the SQLite database file
    ):
        """Initialize storage and create table if needed."""
        self.db_path = db_path
        with sqlite3.connect(self.db_path) as con:
            con.execute(self.SCHEMA)
            con.execute(self.INDEX)

    def save(
        self,
        file_path: str,     # Path to the analyzed audio file
        file_hash: str,     # Hash of the analyzed file in "algo:hexdigest" format
        config_hash: str,   # Hash of the VAD detection config
        ranges: Optional[List[Dict[str, Any]]] = None,  # Detected speech segments
        metadata: Optional[Dict[str, Any]] = None        # VAD metadata
    ) -> None:
        """Save or replace a VAD result (upsert by file_path + config_hash)."""
        with sqlite3.connect(self.db_path) as con:
            con.execute(
                """INSERT OR REPLACE INTO vad_results
                   (file_path, file_hash, config_hash, ranges, metadata, created_at)
                   VALUES (?, ?, ?, ?, ?, ?)""",
                (
                    file_path,
                    file_hash,
                    config_hash,
                    json.dumps(ranges) if ranges else None,
                    json.dumps(metadata) if metadata else None,
                    time.time()
                )
            )

    def save_with_logging(
        self,
        *,
        file_path: str,     # Path to the analyzed audio file
        file_hash: str,     # Hash of the analyzed file in "algo:hexdigest" format
        config_hash: str,   # Hash of the VAD detection config
        ranges: Optional[List[Dict[str, Any]]] = None,  # Detected speech segments
        metadata: Optional[Dict[str, Any]] = None,       # VAD metadata
        logger: Optional[logging.Logger] = None           # Optional logger for success/failure messages
    ) -> bool:  # True if saved; False if the save failed (error logged, not raised)
        """Save a result, logging success/failure. Failures are logged and swallowed (returns False).

        CR-14 follow-up: records a RESULT_SAVED account either way (ok flag +
        file/config references — the journal never carries content) so saves AND
        swallowed save-failures become auditable journal rows."""
        try:
            self.save(
                file_path=file_path,
                file_hash=file_hash,
                config_hash=config_hash,
                ranges=ranges,
                metadata=metadata,
            )
            record_account(SubstrateEventType.RESULT_SAVED.value, {
                "ok": True, "file_path": file_path, "file_hash": file_hash,
                "config_hash": config_hash,
            })
            if logger:
                logger.info(f"Saved VAD result to DB ({file_path})")
            return True
        except Exception as e:
            record_account(SubstrateEventType.RESULT_SAVED.value, {
                "ok": False, "file_path": file_path, "file_hash": file_hash,
                "config_hash": config_hash, "error": str(e),
            })
            if logger:
                logger.error(f"Failed to save to DB: {e}")
            return False

    def get_cached(
        self,
        file_path: str,   # Path to the audio file
        file_hash: str,   # Content hash of the file (cache miss if the file changed)
        config_hash: str  # Config hash to match
    ) -> Optional[VADRow]:  # Cached row or None
        """Retrieve a content-correct cached VAD result.

        Matches on file_path + file_hash + config_hash, so a changed file (new
        file_hash) misses the cache even though a stale row may still exist at the
        same (file_path, config_hash) — the next save() replaces it.

        CR-14 follow-up: a hit records a CACHE_HIT account (the cache-serving
        decision is an account-of-action)."""
        with sqlite3.connect(self.db_path) as con:
            cur = con.execute(
                """SELECT file_path, file_hash, config_hash, ranges, metadata, created_at
                   FROM vad_results WHERE file_path = ? AND file_hash = ? AND config_hash = ?""",
                (file_path, file_hash, config_hash)
            )
            row = cur.fetchone()
            if not row:
                return None
            record_account(SubstrateEventType.CACHE_HIT.value, {
                "file_path": file_path, "file_hash": file_hash,
                "config_hash": config_hash,
            })
            return VADRow(
                file_path=row[0],
                file_hash=row[1],
                config_hash=row[2],
                ranges=json.loads(row[3]) if row[3] else None,
                metadata=json.loads(row[4]) if row[4] else None,
                created_at=row[5]
            )

    def list_jobs(
        self,
        limit: int = 100  # Maximum number of rows to return
    ) -> List[VADRow]:  # List of VAD rows
        """List VAD results ordered by creation time (newest first)."""
        results = []
        with sqlite3.connect(self.db_path) as con:
            cur = con.execute(
                """SELECT file_path, file_hash, config_hash, ranges, metadata, created_at
                   FROM vad_results ORDER BY created_at DESC LIMIT ?""",
                (limit,)
            )
            for row in cur:
                results.append(VADRow(
                    file_path=row[0],
                    file_hash=row[1],
                    config_hash=row[2],
                    ranges=json.loads(row[3]) if row[3] else None,
                    metadata=json.loads(row[4]) if row[4] else None,
                    created_at=row[5]
                ))
        return results

    def verify_file(
        self,
        file_path: str,   # Path to the audio file
        config_hash: str  # Config hash to look up
    ) -> Optional[bool]:  # True if file matches, False if changed, None if not found
        """Verify the analyzed file still matches the hash stored for (file_path, config_hash)."""
        with sqlite3.connect(self.db_path) as con:
            cur = con.execute(
                """SELECT file_hash FROM vad_results
                   WHERE file_path = ? AND config_hash = ?""",
                (file_path, config_hash)
            )
            row = cur.fetchone()
        if not row:
            return None
        return hash_file(file_path) == row[0]

In [ ]:
# get_cached: content-correct lookup + INSERT OR REPLACE upsert on (file_path, config_hash).
import tempfile, os

tmp_db = tempfile.NamedTemporaryFile(suffix=".db", delete=False)
storage = VADStorage(tmp_db.name)

fh = "sha256:" + "a" * 64
cfg = "sha256:" + "c" * 64
storage.save(file_path="/tmp/a.wav", file_hash=fh, config_hash=cfg,
             ranges=[{"start": 0.0, "end": 2.5, "label": "speech", "confidence": 1.0, "payload": {}}],
             metadata={"duration": 6.0, "sample_rate": 16000})

hit = storage.get_cached("/tmp/a.wav", fh, cfg)
assert hit is not None and hit.ranges[0]["end"] == 2.5
# changed content -> miss; different config -> miss
assert storage.get_cached("/tmp/a.wav", "sha256:" + "9" * 64, cfg) is None
assert storage.get_cached("/tmp/a.wav", fh, "sha256:" + "0" * 64) is None

# same (file_path, config), changed content -> replaces the stale row (one row, no orphan)
storage.save(file_path="/tmp/a.wav", file_hash="sha256:" + "b" * 64, config_hash=cfg,
             ranges=[], metadata={})
rows = storage.list_jobs()
assert len(rows) == 1 and rows[0].file_hash == "sha256:" + "b" * 64
print("VADStorage get_cached hit/miss + upsert OK")

In [ ]:
# save_with_logging success + swallowed-failure paths.
test_logger = logging.getLogger("vad_storage_test")
ok = storage.save_with_logging(file_path="/tmp/b.wav", file_hash="sha256:" + "d" * 64,
                               config_hash=cfg, ranges=[], metadata={}, logger=test_logger)
assert ok is True and storage.get_cached("/tmp/b.wav", "sha256:" + "d" * 64, cfg) is not None

_orig = storage.save
def _boom(**kwargs): raise RuntimeError("simulated DB failure")
storage.save = _boom
try:
    ok_fail = storage.save_with_logging(file_path="/tmp/x.wav", file_hash="sha256:" + "e" * 64,
                                        config_hash=cfg, ranges=[], metadata={}, logger=test_logger)
finally:
    storage.save = _orig
assert ok_fail is False
os.unlink(tmp_db.name)
print("VADStorage save_with_logging success + swallowed-failure OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()